# Gold `dim_province`

One row per reporting unit (100): a province, an HUC that DOT reports separately, the City of Isabela, or NCR.

| | |
| --- | --- |
| **Key** | `province_psgc`, the 10-digit PSGC code stored as text |
| **Built from** | `03_silver.geo_reporting_unit`, plus `city_class` from `geo_psgc` |
| **Used by** | `fact_province_year` (foreign key) and the Gold views |

Every check runs before writing. If a `BLOCK` check fails, nothing is written and the notebook raises an error.

## Configuration and helpers

`dq(...)` records each check. `finish(...)` saves the results, stops on any `BLOCK` failure, then writes the table and closes the run log.

In [0]:
from pyspark.sql import functions as F
import uuid
from datetime import datetime

CATALOG = "tuklas_dev"
SILVER_SCHEMA = "03_silver"
GOLD_SCHEMA = "04_gold"
DQ_SCHEMA = "06_data_quality"

NOTEBOOK_NAME = "01_gold_dim_province"      # 02_gold_dim_year / 03_gold_fact_province_year
run_id = f"gold_{NOTEBOOK_NAME[3:]}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

def silver_table(n): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{n}`"
def gold_table(n):   return f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{n}`"
def dq_table(n):     return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{n}`"

dq_results = []

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

spark.sql(f"""
INSERT INTO {gold_table('gold_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"Run ID: {run_id}")

def finish(tables):
    """Save DQ results; stop on any BLOCK failure (nothing written); otherwise overwrite each Gold table and close the run log."""
    dq_df = (spark.createDataFrame(dq_results,
        "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
        "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING")
        .withColumn("checked_at", F.current_timestamp()))
    display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))
    dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
                 "expected", "observed", "sample_keys", "checked_at", "notes") \
         .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))

    blocks = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
    reviews = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]

    def update(status, tw, rw, notes):
        spark.sql(f"""
        UPDATE {gold_table('gold_run_log')}
        SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
            dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
        WHERE run_id = :run_id
        """, args={"status": status, "tw": tw, "rw": rw, "dc": len(dq_results),
                   "db": len(blocks), "dr": len(reviews), "notes": notes, "run_id": run_id})

    if blocks:
        msg = "BLOCK checks failed: " + ", ".join(r[2] for r in blocks) + ". No Gold tables written."
        update("FAILED", 0, 0, msg)
        raise AssertionError(msg + f" See {dq_table('dq_results')} for run_id = '{run_id}'.")

    written = {}
    for name, df in tables.items():
        (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table(name)))
        written[name] = spark.table(gold_table(name)).count()
        print(f"✓ {gold_table(name)}: {written[name]} rows")

    status = "SUCCESS_WITH_REVIEW" if reviews else "SUCCESS"
    notes = (f"{', '.join(f'{k}={v}' for k, v in written.items())}; {len(dq_results)} checks, {len(reviews)} for review"
             + (f" ({', '.join(r[2] for r in reviews)})" if reviews else "") + ".")
    update(status, len(written), sum(written.values()), notes)
    print(f"Run {run_id}: {status}\n{notes}")

## Step 1: Build

`city_class` is not on the unit table, so it is joined from `geo_psgc` on the PSGC code. It is NULL for provinces and NCR.

In [0]:
EXPECTED_UNITS = 100

unit = spark.table(silver_table("geo_reporting_unit"))
psgc = spark.table(silver_table("geo_psgc")).select(F.col("psgc_code").alias("unit_psgc_code"), "city_class")

dim_province = (unit.join(psgc, "unit_psgc_code", "left")
    .select(F.col("unit_psgc_code").alias("province_psgc"),
            F.col("unit_name").alias("province_name"),
            "unit_type",
            F.col("region_code").alias("region_psgc"),
            "region_name",
            "city_class",
            F.col("dot_comparability_note").alias("comparability_note"),
            F.col("psgc_version").alias("psgc_snapshot"),
            "census_population_ref",
            "census_population_ref_year"))
display(dim_province.groupBy("unit_type").count().orderBy("unit_type"))

## Step 2: Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-GLD-01 | 100 units, unique 10-digit codes | BLOCK |
| DQ-GLD-01b | Name, unit type and region filled for every unit | BLOCK |
| DQ-GLD-01c | No replacement characters (`�`) in names | BLOCK |
| DQ-GLD-01d | One PSGC snapshot | BLOCK |

In [0]:
n = dim_province.count()
dup = dim_province.groupBy("province_psgc").count().filter("count > 1").count()
bad_code = dim_province.filter(~F.col("province_psgc").rlike(r"^[0-9]{10}$")).count()
dq("DQ-GLD-01", "dim_province", "BLOCK", dup + bad_code + (0 if n == EXPECTED_UNITS else 1),
   f"{EXPECTED_UNITS} unique 10-digit codes", f"{n} rows, {dup} duplicates, {bad_code} bad codes")

no_name = dim_province.filter(F.col("province_name").isNull() | F.col("region_psgc").isNull() | F.col("unit_type").isNull())
dq("DQ-GLD-01b", "dim_province", "BLOCK", no_name.count(), "name, unit_type and region filled for every unit",
   f"{no_name.count()} incomplete", sample(no_name, "province_psgc"))

bad_chars = dim_province.filter(F.col("province_name").contains("�"))
dq("DQ-GLD-01c", "dim_province", "BLOCK", bad_chars.count(), "0 names with �", f"{bad_chars.count()}")

snap = [r[0] for r in dim_province.select("psgc_snapshot").distinct().collect()]
dq("DQ-GLD-01d", "dim_province", "BLOCK", 0 if len(snap) == 1 else 1, "one PSGC snapshot", f"{snap}")

## Step 3: Write

In [0]:
finish({"dim_province": dim_province})